<a href="https://colab.research.google.com/github/mf2056/Dissertation/blob/main/Baseline_InceptionV3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import numpy as np
import os
import tensorflow as tf
from tensorflow.keras import layers, models

In [4]:
# Load the data
X_train = np.load('/content/drive/MyDrive/X_train.npy')
Y_train_lab = np.load('/content/drive/MyDrive/Y_train_labels.npy')
X_test = np.load('/content/drive/MyDrive/X_test.npy')
Y_test_lab = np.load('/content/drive/MyDrive/Y_test_labels.npy')

In [5]:
# Take only RGB (Bands 0, 1, 2) and scale to [-1, 1] for InceptionV3
X_train_rgb = (X_train[:, :, :, 0:3] * 2) - 1.0
X_test_rgb = (X_test[:, :, :, 0:3] * 2) - 1.0

print(f"Data loaded. Training shape: {X_train_rgb.shape}")

Data loaded. Training shape: (639, 256, 256, 3)


In [6]:
print(f"Train Min: {X_train_rgb.min()}, Train Max: {X_train_rgb.max()}")

Train Min: -1.0, Train Max: 1.0


In [9]:
import numpy as np

# Defining class map (from ESA WorldCover standards)
class_map = {
    10: ("Tree cover", "#006400"),
    20: ("Shrubland", "#ffbb22"),
    30: ("Grassland", "#ffff4c"),
    40: ("Cropland", "#f096ff"),
    50: ("Built-up", "#fa0000"),
    60: ("Bare / Sparse vegetation", "#b4b4b4"),
    70: ("Snow and ice", "#f0f0f0"),
    80: ("Permanent water bodies", "#0064ff"),
    90: ("Herbaceous wetland", "#0096a0"),
}

# Create the internal mapping
unique_labels = sorted(np.unique(Y_train_lab))
label_map = {old: new for new, old in enumerate(unique_labels)}

# Print the Label Mapping Table
print("--- LABEL MAPPING FOR INCEPTIONV3 ---")
for old_id, new_id in sorted(label_map.items(), key=lambda item: item[1]):
    class_name = class_map.get(old_id, ("Unknown", ""))[0]
    print(f"New ID: {new_id:2}  <--  Original ID: {old_id:2} ({class_name})")

# Apply the mapping to create the final training/testing labels
Y_train_ready = np.array([label_map[l] for l in Y_train_lab])
Y_test_ready = np.array([label_map[l] for l in Y_test_lab])

# Data shape verification
print("\n--- DATA SHAPE VERIFICATION ---")
print(f"X_train shape:      {X_train_rgb.shape} (RGB Baseline)")
print(f"Y_train_ready:      {Y_train_ready.shape}")
print(f"Unique classes:     {np.unique(Y_train_ready)}")
print(f"Total Train Samples: {len(Y_train_ready)}")

--- LABEL MAPPING FOR INCEPTIONV3 ---
New ID:  0  <--  Original ID: 10 (Tree cover)
New ID:  1  <--  Original ID: 20 (Shrubland)
New ID:  2  <--  Original ID: 30 (Grassland)
New ID:  3  <--  Original ID: 40 (Cropland)
New ID:  4  <--  Original ID: 50 (Built-up)
New ID:  5  <--  Original ID: 80 (Permanent water bodies)

--- DATA SHAPE VERIFICATION ---
X_train shape:      (639, 256, 256, 3) (RGB Baseline)
Y_train_ready:      (639,)
Unique classes:     [0 1 2 3 4 5]
Total Train Samples: 639


In [37]:
# Define the number of classes based on your unique labels
num_classes = len(np.unique(Y_train_lab))

# Load the base InceptionV3 model (without the top classification layer)
base_model = tf.keras.applications.InceptionV3(
    weights='imagenet',
    include_top=False,
    input_shape=(256, 256, 3)
)

# Freeze the base model (Transfer Learning)
base_model.trainable = False

# Create the custom classification head
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.7), # Prevents overfitting
    layers.Dense(num_classes, activation='softmax') # Final prediction layer
])

# Compile the model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


In [38]:
from sklearn.utils import class_weight

# Calculate weights based on your mapped training labels
weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(Y_train_ready),
    y=Y_train_ready
)

class_weight_dict = dict(enumerate(weights))

In [39]:
import tensorflow as tf
from sklearn.metrics import f1_score
import numpy as np

class TableLogger(tf.keras.callbacks.Callback):
    def __init__(self, val_data):
        super().__init__()
        self.X_val, self.y_val = val_data

    def on_train_begin(self, logs=None):
        print(f"\n{'Epoch':<6} | {'Train Loss':<12} | {'Val Loss':<10} | {'Acc':<8} | {'F1 (Macro)':<10}")
        print("-" * 65)

    def on_epoch_end(self, epoch, logs=None):
        val_logits = self.model.predict(self.X_val, verbose=0)
        val_preds = np.argmax(val_logits, axis=1)
        val_f1 = f1_score(self.y_val, val_preds, average='macro')

        train_loss = logs.get('loss', 0)
        val_loss = logs.get('val_loss', 0)
        val_acc = logs.get('val_accuracy', 0)

        print(f"{epoch+1:<6} | {train_loss:<12.4f} | {val_loss:<10.4f} | {val_acc:<8.4f} | {val_f1:<10.4f}")

In [40]:
# Create the logger with the validation data explicitly provided
table_logger = TableLogger(val_data=(X_test_rgb, Y_test_ready))

# This reduces the learning rate automatically when the model starts "bouncing"
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=3,
    min_lr=0.00001
)

# Run the training
history = model.fit(
    X_train_rgb,
    Y_train_ready,
    validation_data=(X_test_rgb, Y_test_ready),
    epochs=10,
    batch_size=16,
    class_weight=class_weight_dict,
    callbacks=[table_logger, reduce_lr],
    verbose=0
)


Epoch  | Train Loss   | Val Loss   | Acc      | F1 (Macro)
-----------------------------------------------------------------
1      | 1.8666       | 1.3072     | 0.5680   | 0.5384    
2      | 1.3266       | 1.0853     | 0.6640   | 0.6197    
3      | 1.1286       | 0.9414     | 0.6880   | 0.6500    
4      | 0.9865       | 0.9207     | 0.7040   | 0.6643    
5      | 0.8864       | 0.7943     | 0.7200   | 0.7004    
6      | 0.7781       | 0.7688     | 0.7280   | 0.6992    
7      | 0.7361       | 0.7442     | 0.7360   | 0.7077    
8      | 0.6462       | 0.7435     | 0.7360   | 0.7062    
9      | 0.6174       | 0.7202     | 0.7360   | 0.7140    
10     | 0.5938       | 0.6698     | 0.7760   | 0.7532    


In [41]:
from sklearn.metrics import classification_report, accuracy_score, f1_score

# Get Predictions
print("Generating predictions...")
train_preds = np.argmax(model.predict(X_train_rgb), axis=1)
test_preds = np.argmax(model.predict(X_test_rgb), axis=1)

# Print Overall Accuracies
train_acc = accuracy_score(Y_train_ready, train_preds)
test_acc = accuracy_score(Y_test_ready, test_preds)
test_f1_macro = f1_score(Y_test_ready, test_preds, average="macro")

print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy:  {test_acc:.4f}")
print(f"Test Macro F1:  {test_f1_macro:.4f}")

# Detailed Classification Report
target_names = [class_map[old_id][0] for old_id, new_id in sorted(label_map.items(), key=lambda x: x[1])]

print("\n--- DETAILED CLASSIFICATION REPORT (TEST SET) ---")
print(classification_report(Y_test_ready, test_preds, target_names=target_names))

Generating predictions...
20/20 ━━━━━━━━━━━━━━━━━━━━ 7s 384ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step

Train Accuracy: 0.9061
Test Accuracy:  0.7760
Test Macro F1:  0.7532

--- DETAILED CLASSIFICATION REPORT (TEST SET) ---
                        precision    recall  f1-score   support

            Tree cover       0.62      0.83      0.71         6
             Shrubland       0.60      0.69      0.64        13
             Grassland       0.72      0.45      0.55        29
              Cropland       0.83      0.86      0.84        28
              Built-up       0.72      0.95      0.82        22
Permanent water bodies       0.96      0.93      0.94        27

              accuracy                           0.78       125
             macro avg       0.74      0.79      0.75       125
          weighted avg       0.78      0.78      0.77       125

